# 04. Agent công cụ và triển khai API cục bộ
Agent thực thi có định tuyến, KG query, thống kê CSV, calculator, dense/hybrid RAG và FLAN-T5. Đây là tool router xác định; không gắn nhãn nó thành một LLM tự lập kế hoạch.

Notebook khởi động process uvicorn thật trên loopback, gọi HTTP thật, lưu response và log, rồi dừng process. Không tự nhận đã deploy cloud. Swagger UI tại `/docs` được cung cấp khi người dùng khởi động lại dịch vụ.

In [1]:
from pathlib import Path
import sys, json, time, random, platform, importlib.metadata as metadata
ROOT = Path.cwd().resolve()
while not (ROOT / 'data/chapter04').exists():
    if ROOT.parent == ROOT: raise RuntimeError('Không tìm thấy thư mục dự án')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / 'data/chapter04'
RESULT = ROOT / 'results/chapter04'
FIG = ROOT / 'docs/report_chapter_04'
RESULT.mkdir(parents=True, exist_ok=True)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
random.seed(42); np.random.seed(42)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})
environment = {'python': platform.python_version(), 'platform': platform.platform(),
    'packages': {p: metadata.version(p) for p in ['numpy','pandas','networkx','torch','transformers','nbclient','fastapi']}}
display(pd.DataFrame(environment['packages'].items(), columns=['Thư viện','Phiên bản']))

,Thư viện,Phiên bản
0,numpy,2.5.2
1,pandas,3.0.5
2,networkx,3.6.1
3,torch,2.13.0
4,transformers,4.57.6
5,nbclient,0.11.0
6,fastapi,0.141.1


## 1. Kiểm tra công cụ trước triển khai

In [2]:
from src.chapter04.agent import ToolAgent
from src.chapter04.knowledge import load_graph
stats=json.loads((RESULT/'summary.json').read_text(encoding='utf-8'))['uci']
graph=load_graph(ROOT/'knowledge_graph/chapter04/finance_graph.json.gz')
tool_agent=ToolAgent(None,stats,{'finance':graph})
samples=[
    tool_agent.ask('How many encounters were readmitted within 30 days?','diabetes'),
    tool_agent.ask('Revenue growth','finance',task='calculate',operation='growth',values=[100,120],units=['USD','USD']),
    tool_agent.ask('Find Visa reports','finance',task='graph',company='V')]
display(pd.DataFrame([{'status':x['status'],'answer':x['answer'],'trace':x['trace']} for x in samples]))

,status,answer,trace
0,computed,11357,"[{'tool': 'dataset_statistics', 'result': 11357}]"
1,computed,20.0,"[{'tool': 'financial_calculator', 'inputs': {'..."
2,queried,2 reports in indexed collection,"[{'tool': 'query_graph', 'company': 'V', 'repo..."


## 2. Khởi động API thật, gọi HTTP và thu bằng chứng

In [3]:
import subprocess, socket, requests, os
with socket.socket() as sock:
    sock.bind(('127.0.0.1',0));port=sock.getsockname()[1]
base=f'http://127.0.0.1:{port}'
log_path=RESULT/'agent_api_server.log'
requests_to_run=[
    {'domain':'diabetes','question':'How many encounters were readmitted within 30 days?','task':'statistics'},
    {'domain':'finance','question':'Revenue growth','task':'calculate','operation':'growth','values':[100,120],'units':['USD','USD']},
    {'domain':'finance','question':'Find Visa reports','task':'graph','company':'V'},
    {'domain':'diabetes','question':'What organ makes insulin?','task':'rag'},
    {'domain':'finance','question':'Invalid divide','task':'calculate','operation':'divide','values':[1,0]},
    {'domain':'invalid','question':'wrong domain'}]
api_records=[]
with log_path.open('w',encoding='utf-8') as log:
    flags=subprocess.CREATE_NO_WINDOW if os.name=='nt' else 0
    process=subprocess.Popen([sys.executable,'-m','uvicorn','deployment.chapter04.app:app','--host','127.0.0.1','--port',str(port)],cwd=ROOT,stdout=log,stderr=log,creationflags=flags)
    try:
        deadline=time.monotonic()+120
        while time.monotonic()<deadline:
            if process.poll() is not None:raise RuntimeError(log_path.read_text(encoding='utf-8'))
            try:
                health=requests.get(base+'/health',timeout=2)
                if health.status_code==200:break
            except requests.RequestException:pass
            time.sleep(.5)
        else:raise TimeoutError('API did not become healthy')
        print('HTTP GET /health:',health.json())
        swagger=requests.get(base+'/docs',timeout=10)
        assert swagger.status_code==200 and 'SwaggerUIBundle' in swagger.text
        for body in requests_to_run:
            started=time.perf_counter();response=requests.post(base+'/api/v1/knowledge/ask',json=body,timeout=180)
            api_records.append({'request':body,'http_status':response.status_code,'response':response.json(),'round_trip_seconds':time.perf_counter()-started})
            print('HTTP',response.status_code,body['question'],flush=True)
    finally:
        process.terminate()
        try:process.wait(timeout=30)
        except subprocess.TimeoutExpired:process.kill();process.wait()
assert [x['http_status'] for x in api_records]==[200,200,200,200,422,422]
assert api_records[0]['response']['value']==11357
assert api_records[1]['response']['value']==20
assert api_records[3]['response']['citation_ids_valid']
display(pd.DataFrame([{'question':x['request']['question'],'HTTP':x['http_status'],'answer':x['response'].get('answer',x['response'].get('detail')),'seconds':x['round_trip_seconds']} for x in api_records]))

HTTP GET /health: {'status': 'ok', 'mode': 'local CPU', 'generator': 'google/flan-t5-small', 'agent': 'deterministic tool router'}


HTTP 200 How many encounters were readmitted within 30 days?


HTTP 200 Revenue growth


HTTP 200 Find Visa reports


HTTP 200 What organ makes insulin?


HTTP 422 Invalid divide


HTTP 422 wrong domain


,question,HTTP,answer,seconds
0,How many encounters were readmitted within 30 ...,200,11357,0.392848
1,Revenue growth,200,20.0,0.018200
2,Find Visa reports,200,2 reports in indexed collection,0.029897
3,What organ makes insulin?,200,pancreas,119.462707
4,Invalid divide,422,Division by zero,0.009304
5,wrong domain,422,"[{'type': 'literal_error', 'loc': ['body', 'do...",0.007799


## 3. Trace thật của Agent và kiểm tra nguồn

In [4]:
rag_response=api_records[3]['response']
display(pd.DataFrame(rag_response['trace']))
display(pd.DataFrame(rag_response['citations']))
print('Generated answer:',rag_response['answer'])
print('Status:',rag_response['status'],'; Citation ID check is not semantic correctness validation.')
deployment_result={'health':health.json(),'swagger_status':swagger.status_code,'requests':api_records,
    'deployment':'real local uvicorn HTTP process, stopped after verification','host':'127.0.0.1','port':port,
    'agent_type':'deterministic bounded tool router','environment':environment}
(RESULT/'agent_deployment_results.json').write_text(json.dumps(deployment_result,ensure_ascii=False,indent=2),encoding='utf-8')

,tool,domain,count,model,valid,semantic_support_checked
0,search_documents,diabetes,3.0,NaN,NaN,NaN
1,local_generator,NaN,NaN,google/flan-t5-small,NaN,NaN
2,verify_citation_ids,NaN,NaN,NaN,True,False


,id,url,locator
0,what-is-diabetes:14,https://www.niddk.nih.gov/health-information/d...,what-is-diabetes:14
1,symptoms-causes:54,https://www.niddk.nih.gov/health-information/d...,symptoms-causes:54
2,what-is-diabetes:9,https://www.niddk.nih.gov/health-information/d...,what-is-diabetes:9


Generated answer: pancreas
Status: generated_unverified ; Citation ID check is not semantic correctness validation.


8192

## 4. Chạy lại dịch vụ để trình diễn
Tại thư mục gốc dự án:
```powershell
.venv/Scripts/python.exe -m uvicorn deployment.chapter04.app:app --host 127.0.0.1 --port 8044
```
Mở http://127.0.0.1:8044/docs. Không cần khóa API, corpus và model đều tải trong dự án. Calculator nhận operands có cấu trúc; các con số đó do caller đưa vào, chưa tự trích từ câu hỏi. Mô hình nhỏ có thể trả lời sai; API đánh dấu `generated_unverified` cho đầu ra sinh.

Dockerfile đã chuẩn bị nhưng kiểm chứng triển khai trong notebook là HTTP cục bộ. Không công bố SLA, đánh giá lâm sàng hoặc lợi nhuận chứng khoán.